In [ ]:
# XGBoost
# Gradient Boost

# Business Case :

### Based on given features we need to find whether an employee will leave the company or not.

In [ ]:
#!pip install ydata-profiling

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from ydata_profiling import ProfileReport
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder,OrdinalEncoder
import warnings
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import pickle
from sklearn.pipeline import Pipeline
%matplotlib inline
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'ydata_profiling'

In [ ]:
df=pd.read_csv("HR.csv")

# BASIC CHECKS

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Select numerical columns
numerical_cols = df.select_dtypes(include=['int', 'float']).columns

# Create box plots for all numerical features
plt.figure(figsize=(12, 8))
df[numerical_cols].boxplot()
plt.xticks(rotation=90)
plt.title('Box plots for Numerical Features')
plt.ylabel('Value')
plt.xlabel('Feature')
plt.show()


In [ ]:
df.info()

- There are no null values in the dataset.
- Attrition is the target column

In [ ]:
df.describe()

In [ ]:
df.describe(include = "O")

# EDA

In [ ]:
profile=ProfileReport(df,title="EDA",explorative=True)

In [ ]:
profile

# INSIGHTS FROM EDA 

### Write their own insights.

# Correlation

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(28,28))#increase plot size
sns.heatmap(df.select_dtypes(include=['int', 'float']).corr(),cmap="RdYlGn",annot=True)

In [ ]:
df.corr()

In [ ]:
np.where(df.select_dtypes(include=['int', 'float']).corr()>0.9)

- There is a high correlation between JobLevel and MonthlyIncome hence the MonthlyIncome column can be dropped.

# PRE-PROCESSING

In [ ]:
df.columns

**The below columns are numerical columns having continuous values and hence we will apply Standard Scaler for all these columns**
- Age 
- DailyRate
- DistanceFromHome
- MonthlyRate
- HourlyRate
- NumCompaniesWorked
- TotalWorkingYears
- TrainingTimesLastYear
- YearsAtCompany
- YearsInCurrentRole
- YearsSinceLastPromotion
- YearsWithCurrManager
- PercentSalaryHike 

In [ ]:
df[['Age', 'DailyRate', 'DistanceFromHome', 'MonthlyRate', 'HourlyRate', 'NumCompaniesWorked', 'TotalWorkingYears', 
    'TrainingTimesLastYear', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager','PercentSalaryHike']]

# BusinessTravel :

In [ ]:
df.BusinessTravel.value_counts()

- We will use One Hot Encoder for this column.

# Department :

In [ ]:
df.Department.value_counts()

- We will use One hot encoder for this column.

# Education :

In [ ]:
df.Education.value_counts()

- The values are already arranged ordinally so we don't need to do anything.

# EducationField :

In [ ]:
df.EducationField.value_counts()

- We will use One Hot Encoder.


# EnvironmentSatisfaction :

In [ ]:
df.EnvironmentSatisfaction.value_counts()

- The values were already arranged ordinally so we don't have to do anything.

# Gender :

In [ ]:
df.Gender.value_counts()

- We will apply One Hot Encoding to this column

# JobInvolvement :

In [ ]:
df.JobInvolvement.value_counts()

- The values are already arranged in an orderly manner.

# JobLevel :

In [ ]:
df.JobLevel.value_counts()

- The values are already arranged in an ordinal manner.

# JobRole :

In [ ]:
df.JobRole.value_counts()

- We will use one hot encoding.

# JobSatisfaction :

In [ ]:
df.JobSatisfaction.value_counts()

- This column is already arranged in an ordinal manner.

# MaritalStatus :

In [ ]:
df.MaritalStatus.value_counts()

We will use One Hot Encoder for this column.

# OverTime :

In [ ]:
df.OverTime.value_counts()

- We will use Label encoder for this column.


# PerformanceRating :

In [ ]:
df.PerformanceRating.value_counts()

- There are no changes to be made in this column since it is already present in an orderly manner.

# RelationshipSatisfaction :

In [ ]:
df.RelationshipSatisfaction.value_counts()

- This column is already arranged in an ordinal manner.

# StockOptionLevel :

In [ ]:
df.StockOptionLevel.value_counts()

- This column is already arranged in an ordinal manner.

# TrainingTimesLastYear :

In [ ]:
df.TrainingTimesLastYear.value_counts()

- This column is already arranged in an ordinal manner.

# WorkLifeBalance :

In [ ]:
df.WorkLifeBalance.value_counts()

- This column is already arranged in an ordinal manner.

**All the columns that have a prior hierarchical order will be left as it is and Ordinal encoder will not be applied to such columns.**

# Creating the preprocessing pipeline :

In [ ]:
len(df.columns)

In [ ]:
OHE_columns=['BusinessTravel','Department','MaritalStatus','EducationField','Gender','JobRole']
standard_scaler=['Age', 'DailyRate', 'DistanceFromHome','MonthlyRate', 'HourlyRate', 'NumCompaniesWorked', 'TotalWorkingYears', 
                 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager','PercentSalaryHike']
label_encoder=['OverTime']
passthrough=['Education','EnvironmentSatisfaction','JobInvolvement','JobLevel','JobSatisfaction','PercentSalaryHike','RelationshipSatisfaction',
             'StockOptionLevel','TrainingTimesLastYear','WorkLifeBalance','PerformanceRating']

# Steps of preprocesing for features:
- **LabelEncoder for**   :  'OverTime'
- **One hot encoder for**:  'BusinessTravel','Department','MaritalStatus','EducationField','Gender','JobRole'
- **Standard scaler for**:  'Age', 'DailyRate', 'DistanceFromHome', 'MonthlyIncome', 'MonthlyRate', 'HourlyRate', 'NumCompaniesWorked',
                            'TotalWorkingYears','TrainingTimesLastYear', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
                            'YearsWithCurrManager','PercentSalaryHike'
- **Pass through for**:     'Education','EnvironmentSatisfaction','JobInvolvement','JobLevel','JobSatisfaction','PercentSalaryHike',
                            'RelationshipSatisfaction','StockOptionLevel','TrainingTimesLastYear','WorkLifeBalance','PerformanceRating'


In [ ]:
class ModifiedLabelEncoder(LabelEncoder):
    def fit_transform(self, y, *args, **kwargs):
        return super().fit_transform(y).reshape(-1, 1)

    def transform(self, y, *args, **kwargs):
        return super().transform(y).reshape(-1, 1)

In [ ]:
def same(x):
    return x

In [ ]:
no_trans=FunctionTransformer(same)

In [ ]:
preprocessor = ColumnTransformer([
    ("OHE columns", OneHotEncoder(), OHE_columns),
    ("Label_encoder", ModifiedLabelEncoder(), label_encoder),
    ("Standard_scaler", StandardScaler(), standard_scaler),
    ('Pass_through',no_trans,passthrough)])

In [ ]:
preprocessor

In [ ]:
file=open("DecisionTree_RandomForest.pkl","wb")

In [ ]:
pickle.dump(preprocessor,file)